# AI-Generated War Video — Low-Level Forensic Analysis

Signal-level analysis to find how AI-generated videos differ from real footage — without relying on faces or semantic content.

**4 analysis layers:**
1. **Spatial frequency** — power spectrum slope (1/f² law)
2. **Temporal consistency** — frame delta statistics
3. **Noise residue** — camera PRNU fingerprint
4. **DCT distribution** — compression domain statistics

---
**Videos needed (in your Google Drive):**
- 2 real videos (verified outdoor/conflict footage)
- 2 AI-generated videos

Put them all in one folder in Drive, e.g. `My Drive/forensics_videos/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted")

Mounted at /content/drive
Drive mounted


In [ ]:
import os

REAL_VIDEO_1 = "/content/drive/MyDrive/analysis_videos/real_1.mp4"
REAL_VIDEO_2 = "/content/drive/MyDrive/analysis_videos/real_2.mp4"
AI_VIDEO_1   = "/content/drive/MyDrive/analysis_videos/ai_1.mp4"
AI_VIDEO_2   = "/content/drive/MyDrive/analysis_videos/ai_2.mp4"

videos = {
    "Real 1":  REAL_VIDEO_1,
    "Real 2":  REAL_VIDEO_2,
    "AI 1":    AI_VIDEO_1,
    "AI 2":    AI_VIDEO_2,
}

print("Checking files...")
all_ok = True
for name, path in videos.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    status = f"{size_mb:.1f} MB" if exists else " NOT FOUND"
    print(f"  {name}: {status}  →  {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n All videos found. Proceed to Cell 3.")
else:
    print("\n  Fix the paths above before continuing.")

Checking files...
  Real 1: 27.7 MB  →  /content/drive/MyDrive/analysis_videos/real_1.mp4
  Real 2: 6.0 MB  →  /content/drive/MyDrive/analysis_videos/real_2.mp4
  AI 1: 1.5 MB  →  /content/drive/MyDrive/analysis_videos/ai_1.mp4
  AI 2: 1.5 MB  →  /content/drive/MyDrive/analysis_videos/ai_2.mp4

 All videos found. Proceed to Cell 3.


In [ ]:
!pip install -q scikit-image opencv-python-headless scipy matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import fftpack, stats
from skimage.restoration import denoise_wavelet
import warnings
warnings.filterwarnings("ignore")

def extract_frames(video_path, max_frames=60):
    """Extract evenly spaced frames from a video."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"  {os.path.basename(video_path)}: {w}x{h}, {fps:.1f}fps, {total} frames ({total/max(fps,1):.1f}s)")

    indices = np.linspace(0, total - 1, min(max_frames, total), dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame.astype(np.float32))
    cap.release()
    return frames

def to_gray(frames):
    return [cv2.cvtColor(f.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
            for f in frames]

print("✅ Utilities loaded")

## Cell 5 — Extract All Frames
This copies video data from Drive into Colab memory. May take 1-2 min depending on file size.

In [ ]:
print("Extracting frames from all 4 videos...\n")
all_frames = {}
for name, path in videos.items():
    all_frames[name] = extract_frames(path, max_frames=60)

print(f"\n✅ Done. Frames in memory: { {k: len(v) for k, v in all_frames.items()} }")

## Cell 6 — Layer 1: Spatial Power Spectrum

**Physics:** Natural images follow a 1/f² power law. AI generators (DiT + VAE decoder) underpower high spatial frequencies due to spectral bias. On a log-log plot, AI videos have a steeper slope — they're missing fine detail energy.

In [ ]:
def radial_power_spectrum(gray_frame, n_bins=100):
    """Radially averaged 1D power spectrum of a grayscale frame."""
    h, w = gray_frame.shape
    size = min(h, w)
    crop = gray_frame[:size, :size]

    fft2d     = fftpack.fft2(crop)
    fft_shift = fftpack.fftshift(fft2d)
    power_2d  = np.abs(fft_shift) ** 2

    cy, cx = size // 2, size // 2
    y_i, x_i = np.indices((size, size))
    r = np.sqrt((x_i - cx)**2 + (y_i - cy)**2).astype(int)

    max_r = size // 2
    radial = np.array([
        power_2d[r == ri].mean() if (r == ri).any() else 0
        for ri in range(1, max_r)
    ])

    bin_edges = np.linspace(0, len(radial), n_bins + 1, dtype=int)
    freq  = np.array([bin_edges[i] for i in range(n_bins)])
    power = np.array([radial[bin_edges[i]:bin_edges[i+1]].mean() for i in range(n_bins)])

    return np.log10(freq + 1), np.log10(power + 1e-10)

def mean_power_spectrum(frames):
    gray = to_gray(frames)
    specs = [radial_power_spectrum(g) for g in gray]
    return specs[0][0], np.mean([s[1] for s in specs], axis=0)

# Compute for all 4 videos
psd_results = {}
for name, frames in all_frames.items():
    freq, psd = mean_power_spectrum(frames)
    slope, intercept, *_ = stats.linregress(freq[5:], psd[5:])
    psd_results[name] = {"freq": freq, "psd": psd, "slope": slope}
    print(f"  {name}: spectral slope = {slope:.3f}")

print("\n📊 Natural images: slope typically between -2.0 and -2.8")
print("   AI generators:  slope typically steeper (more negative, e.g. -3.5)")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor="#0d0d0d")
fig.suptitle("Layer 1 — Radial Power Spectrum (log-log)", color="white", fontsize=14, fontweight="bold")

colors = {"Real 1": "#00e5ff", "Real 2": "#00bfa5", "AI 1": "#ff4081", "AI 2": "#ff6d00"}
PANEL  = "#1a1a2e"
TEXT   = "#e0e0e0"

ax = axes[0]
ax.set_facecolor(PANEL)
for name, res in psd_results.items():
    ax.plot(res["freq"], res["psd"], color=colors[name], lw=2,
            label=f"{name} (slope={res['slope']:.2f})")
ax.set_xlabel("log₁₀(Spatial Frequency)", color=TEXT)
ax.set_ylabel("log₁₀(Power)", color=TEXT)
ax.tick_params(colors=TEXT)
ax.set_title("Power Spectrum — All Videos", color=TEXT)
ax.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor("#333355")

ax2 = axes[1]
ax2.set_facecolor(PANEL)
slopes = [psd_results[n]["slope"] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]]
bars = ax2.bar(["Real 1", "Real 2", "AI 1", "AI 2"], slopes,
               color=[colors[n] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]], alpha=0.85)
ax2.axhspan(-2.8, -2.0, color="#00e5ff", alpha=0.08, label="Natural image range")
ax2.set_ylabel("Spectral Slope", color=TEXT)
ax2.set_title("Slope Comparison\n(shaded = natural 1/f² range)", color=TEXT)
ax2.tick_params(colors=TEXT)
for spine in ax2.spines.values(): spine.set_edgecolor("#333355")
ax2.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for bar, val in zip(bars, slopes):
    ax2.text(bar.get_x() + bar.get_width()/2, val - 0.05, f"{val:.2f}",
             ha="center", va="top", color=TEXT, fontsize=9)

plt.tight_layout()
plt.savefig("/content/layer1_power_spectrum.png", dpi=150, bbox_inches="tight", facecolor="#0d0d0d")
plt.show()
print("💾 Saved: layer1_power_spectrum.png")

## Cell 7 — Layer 2: Temporal Consistency

**Physics:** Real footage has physically driven, chaotic frame-to-frame changes (camera shake, turbulence, real motion). AI DiT models temporally regularize the latent space — producing transitions that are *too smooth* compared to real physics. We measure the coefficient of variation (CV) of the frame delta signal.

In [ ]:
def frame_delta_signal(frames):
    """Mean absolute pixel difference between consecutive frames."""
    deltas = []
    for i in range(1, len(frames)):
        diff = np.abs(frames[i] - frames[i-1])
        deltas.append(diff.mean())
    return np.array(deltas)

temporal_results = {}
for name, frames in all_frames.items():
    d = frame_delta_signal(frames)
    cv = d.std() / (d.mean() + 1e-8)
    temporal_results[name] = {"deltas": d, "mean": d.mean(), "std": d.std(), "cv": cv}
    print(f"  {name}: mean_delta={d.mean():.2f}  std={d.std():.2f}  CV={cv:.4f}")

print("\n📊 Higher CV = more physically random motion (real)")
print("   Lower CV  = over-regularized, suspiciously smooth (AI)")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor="#0d0d0d")
fig.suptitle("Layer 2 — Temporal Consistency Analysis", color="white", fontsize=14, fontweight="bold")

# Delta signal over time
ax = axes[0]
ax.set_facecolor(PANEL)
for name, res in temporal_results.items():
    ax.plot(res["deltas"], color=colors[name], lw=1.5, alpha=0.9, label=name)
ax.set_xlabel("Frame index", color=TEXT)
ax.set_ylabel("Mean |ΔPixel|", color=TEXT)
ax.set_title("Frame Delta Over Time", color=TEXT)
ax.tick_params(colors=TEXT)
ax.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor("#333355")

# CV bar chart
ax2 = axes[1]
ax2.set_facecolor(PANEL)
cvs = [temporal_results[n]["cv"] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]]
bars = ax2.bar(["Real 1", "Real 2", "AI 1", "AI 2"], cvs,
               color=[colors[n] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]], alpha=0.85)
ax2.set_ylabel("Coefficient of Variation", color=TEXT)
ax2.set_title("Temporal Variability (CV)\nHigher = more physically random", color=TEXT)
ax2.tick_params(colors=TEXT)
for spine in ax2.spines.values(): spine.set_edgecolor("#333355")
for bar, val in zip(bars, cvs):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.3f}",
             ha="center", va="bottom", color=TEXT, fontsize=9)

# Temporal frequency spectrum
ax3 = axes[2]
ax3.set_facecolor(PANEL)
for name, res in temporal_results.items():
    d = res["deltas"] - res["deltas"].mean()
    fft = np.abs(np.fft.rfft(d)) ** 2
    freqs = np.fft.rfftfreq(len(d))
    ax3.semilogy(freqs[1:], fft[1:], color=colors[name], lw=1.5, label=name)
ax3.set_xlabel("Temporal Frequency", color=TEXT)
ax3.set_ylabel("Power (log)", color=TEXT)
ax3.set_title("Temporal Frequency Spectrum\nReal=broadband, AI=low-freq dominant", color=TEXT)
ax3.tick_params(colors=TEXT)
ax3.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for spine in ax3.spines.values(): spine.set_edgecolor("#333355")

plt.tight_layout()
plt.savefig("/content/layer2_temporal.png", dpi=150, bbox_inches="tight", facecolor="#0d0d0d")
plt.show()
print("💾 Saved: layer2_temporal.png")

## Cell 8 — Layer 3: Noise Residue / PRNU

**Physics:** Real cameras have a fixed spatial noise pattern (PRNU — Photo Response Non-Uniformity) from sensor manufacturing imperfections. Average the noise residue across frames and it *emerges* from the average. AI video has no physical sensor — the noise residue is purely stochastic per-frame, so inter-frame correlation is near zero.

⚠️ This cell uses wavelet denoising — takes ~3-5 minutes.

In [ ]:
def noise_residue(gray_frame):
    """Extract noise residue via wavelet denoising."""
    norm = gray_frame / 255.0
    denoised = denoise_wavelet(norm, method='BayesShrink', mode='soft',
                               wavelet_levels=4, channel_axis=None)
    return norm - denoised

def prnu_correlation(frames, n=20):
    """Inter-frame noise residue correlation. High = real camera, Near-zero = AI."""
    gray = to_gray(frames[:n])
    residues = [noise_residue(g).flatten() for g in gray]
    corrs = []
    for i in range(0, len(residues) - 1, 3):
        r, _ = stats.pearsonr(residues[i], residues[i+1])
        corrs.append(r)
    return float(np.mean(corrs))

def prnu_map(frames, n=15):
    """Average noise residue map — PRNU pattern should emerge for real cameras."""
    gray = to_gray(frames[:n])
    residues = np.array([noise_residue(g) for g in gray])
    return residues.mean(axis=0)

prnu_results = {}
for name, frames in all_frames.items():
    print(f"  Processing {name}...", end=" ")
    corr = prnu_correlation(frames, n=15)
    pmap = prnu_map(frames, n=12)
    prnu_results[name] = {"correlation": corr, "map": pmap}
    print(f"inter-frame correlation = {corr:+.4f}")

print("\n📊 Higher correlation = stable PRNU fingerprint = real camera")
print("   Near-zero = random per-frame noise = AI generation")

# Plot
fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor="#0d0d0d")
fig.suptitle("Layer 3 — Noise Residue / PRNU Analysis", color="white", fontsize=14, fontweight="bold")

for col, name in enumerate(["Real 1", "Real 2", "AI 1", "AI 2"]):
    res = prnu_results[name]

    # Top row: PRNU map
    ax = axes[0][col]
    pmap = res["map"]
    pmap_norm = (pmap - pmap.min()) / (pmap.max() - pmap.min() + 1e-8)
    ax.imshow(pmap_norm, cmap="hot", aspect="auto")
    ax.set_title(f"{name}\nPRNU Map", color=colors[name], fontsize=9)
    ax.axis("off")

    # Bottom row: noise histogram
    ax2 = axes[1][col]
    ax2.set_facecolor(PANEL)
    flat = res["map"].flatten()
    ax2.hist(flat, bins=80, color=colors[name], alpha=0.8, density=True)
    ax2.set_title(f"Noise Distribution\ncorr={res['correlation']:+.4f}", color=TEXT, fontsize=9)
    ax2.tick_params(colors=TEXT, labelsize=7)
    for spine in ax2.spines.values(): spine.set_edgecolor("#333355")

plt.tight_layout()
plt.savefig("/content/layer3_prnu.png", dpi=150, bbox_inches="tight", facecolor="#0d0d0d")
plt.show()
print("💾 Saved: layer3_prnu.png")

## Cell 9 — Layer 4: DCT Coefficient Distribution

**Physics:** H.264/HEVC compression operates in the DCT domain. Real footage compressed this way produces DCT coefficients following a specific Laplacian distribution (heavy tails, high kurtosis). AI content — even after re-compression — has different underlying signal statistics, producing a different tail behavior.

In [ ]:
def dct_coefficients(frames, n_frames=8):
    """Collect all AC DCT coefficients from 8x8 blocks across frames."""
    all_coeffs = []
    for frame in frames[:n_frames]:
        gray = cv2.cvtColor(frame.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
        h, w = gray.shape
        for y in range(0, h - 8, 16):   # stride 16 for speed
            for x in range(0, w - 8, 16):
                block = gray[y:y+8, x:x+8]
                dct_block = cv2.dct(block)
                ac = dct_block.flatten()[1:]  # skip DC (index 0,0)
                all_coeffs.extend(ac.tolist())
    return np.array(all_coeffs)

dct_results = {}
for name, frames in all_frames.items():
    coeffs = dct_coefficients(frames, n_frames=8)
    kurt   = float(stats.kurtosis(coeffs))
    skew   = float(stats.skew(coeffs))
    scale  = float(np.abs(coeffs).mean())  # Laplacian scale = mean(|x|)
    dct_results[name] = {"coeffs": coeffs, "kurtosis": kurt, "skew": skew, "scale": scale}
    print(f"  {name}: kurtosis={kurt:.2f}  skew={skew:.3f}  Laplacian scale={scale:.3f}")

print("\n📊 Real footage (after codec compression): typically higher kurtosis (heavier tails)")
print("   AI content: different kurtosis/scale due to VAE reconstruction statistics")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor="#0d0d0d")
fig.suptitle("Layer 4 — DCT Coefficient Distribution", color="white", fontsize=14, fontweight="bold")

# Density curves
ax = axes[0]
ax.set_facecolor(PANEL)
x_range = np.linspace(-60, 60, 300)
for name, res in dct_results.items():
    sample = np.clip(res["coeffs"][::200], -300, 300)
    kde = stats.gaussian_kde(sample)
    ax.plot(x_range, kde(x_range), color=colors[name], lw=2,
            label=f"{name} (kurt={res['kurtosis']:.1f})")
ax.set_xlabel("DCT Coefficient Value", color=TEXT)
ax.set_ylabel("Density", color=TEXT)
ax.set_title("DCT AC Coefficient Density", color=TEXT)
ax.tick_params(colors=TEXT)
ax.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor("#333355")

# Kurtosis bar
ax2 = axes[1]
ax2.set_facecolor(PANEL)
kurts = [dct_results[n]["kurtosis"] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]]
bars = ax2.bar(["Real 1", "Real 2", "AI 1", "AI 2"], kurts,
               color=[colors[n] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]], alpha=0.85)
ax2.set_ylabel("Kurtosis", color=TEXT)
ax2.set_title("DCT Kurtosis\n(tail heaviness)", color=TEXT)
ax2.tick_params(colors=TEXT)
for spine in ax2.spines.values(): spine.set_edgecolor("#333355")
for bar, val in zip(bars, kurts):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.5, f"{val:.1f}",
             ha="center", va="bottom", color=TEXT, fontsize=9)

# Log-scale tails (most revealing)
ax3 = axes[2]
ax3.set_facecolor(PANEL)
for name, res in dct_results.items():
    sample = np.clip(res["coeffs"][::200], -300, 300)
    kde = stats.gaussian_kde(sample)
    ax3.semilogy(x_range, kde(x_range), color=colors[name], lw=2, label=name)
ax3.set_xlabel("DCT Coefficient Value", color=TEXT)
ax3.set_ylabel("Density (log scale)", color=TEXT)
ax3.set_title("DCT Tails (log scale)\nDivergence here = compression mismatch", color=TEXT)
ax3.tick_params(colors=TEXT)
ax3.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for spine in ax3.spines.values(): spine.set_edgecolor("#333355")

plt.tight_layout()
plt.savefig("/content/layer4_dct.png", dpi=150, bbox_inches="tight", facecolor="#0d0d0d")
plt.show()
print("💾 Saved: layer4_dct.png")

## Cell 10 — Final Summary Dashboard
All 4 metrics in one figure. This is the one to use for LinkedIn.

In [ ]:
fig = plt.figure(figsize=(20, 10), facecolor="#0d0d0d")
fig.suptitle("AI-Generated vs Real War Video — Forensic Signal Analysis",
             color="white", fontsize=16, fontweight="bold", y=1.01)

gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.38)

metric_names  = ["Spectral Slope\n(closer to -2.5 = real)",
                 "Temporal CV\n(higher = more physical)",
                 "PRNU Correlation\n(higher = camera fingerprint)",
                 "DCT Kurtosis\n(signal of codec stats)"]

metric_values = {
    "Real 1": [psd_results["Real 1"]["slope"],
               temporal_results["Real 1"]["cv"],
               prnu_results["Real 1"]["correlation"],
               dct_results["Real 1"]["kurtosis"]],
    "Real 2": [psd_results["Real 2"]["slope"],
               temporal_results["Real 2"]["cv"],
               prnu_results["Real 2"]["correlation"],
               dct_results["Real 2"]["kurtosis"]],
    "AI 1":   [psd_results["AI 1"]["slope"],
               temporal_results["AI 1"]["cv"],
               prnu_results["AI 1"]["correlation"],
               dct_results["AI 1"]["kurtosis"]],
    "AI 2":   [psd_results["AI 2"]["slope"],
               temporal_results["AI 2"]["cv"],
               prnu_results["AI 2"]["correlation"],
               dct_results["AI 2"]["kurtosis"]],
}

# Top row: bar charts for each metric
for i, metric in enumerate(metric_names):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(PANEL)
    vals = [metric_values[n][i] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]]
    bar_colors = [colors[n] for n in ["Real 1", "Real 2", "AI 1", "AI 2"]]
    bars = ax.bar(["R1", "R2", "AI1", "AI2"], vals, color=bar_colors, alpha=0.85)
    ax.set_title(metric, color=TEXT, fontsize=9)
    ax.tick_params(colors=TEXT, labelsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor("#333355")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + abs(bar.get_height()) * 0.02,
                f"{val:.2f}", ha="center", va="bottom", color=TEXT, fontsize=8)

# Bottom row: power spectrum overlay (most visual)
ax_psd = fig.add_subplot(gs[1, :2])
ax_psd.set_facecolor(PANEL)
for name, res in psd_results.items():
    ls = "--" if "AI" in name else "-"
    ax_psd.plot(res["freq"], res["psd"], color=colors[name], lw=2, ls=ls,
                label=f"{name} (slope={res['slope']:.2f})")
ax_psd.set_xlabel("log₁₀(Spatial Frequency)", color=TEXT)
ax_psd.set_ylabel("log₁₀(Power)", color=TEXT)
ax_psd.set_title("Radial Power Spectrum — Real (solid) vs AI (dashed)\nDivergence at right = AI missing high-frequency energy",
                 color=TEXT, fontsize=10)
ax_psd.tick_params(colors=TEXT)
ax_psd.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for spine in ax_psd.spines.values(): spine.set_edgecolor("#333355")

# Bottom right: temporal delta overlay
ax_t = fig.add_subplot(gs[1, 2:])
ax_t.set_facecolor(PANEL)
for name, res in temporal_results.items():
    ls = "--" if "AI" in name else "-"
    ax_t.plot(res["deltas"], color=colors[name], lw=1.5, ls=ls, alpha=0.9,
              label=f"{name} (CV={res['cv']:.3f})")
ax_t.set_xlabel("Frame index", color=TEXT)
ax_t.set_ylabel("Mean |ΔPixel|", color=TEXT)
ax_t.set_title("Frame Delta Signal — Real (solid) vs AI (dashed)\nAI = flatter, over-regularized motion",
               color=TEXT, fontsize=10)
ax_t.tick_params(colors=TEXT)
ax_t.legend(facecolor="#111130", labelcolor=TEXT, fontsize=9)
for spine in ax_t.spines.values(): spine.set_edgecolor("#333355")

plt.savefig("/content/forensic_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor="#0d0d0d")
plt.show()
print("💾 Saved: forensic_dashboard.png — use this for LinkedIn")

## Cell 11 — Print Full Summary Table

In [ ]:
print("\n" + "="*70)
print("FORENSIC SUMMARY — ALL 4 METRICS")
print("="*70)
print(f"{'Video':<12} {'PSD Slope':>12} {'Temporal CV':>13} {'PRNU Corr':>12} {'DCT Kurt':>10}")
print("-"*70)
for name in ["Real 1", "Real 2", "AI 1", "AI 2"]:
    tag = "[REAL]" if "Real" in name else "[AI]  "
    print(f"{tag} {name:<6} "
          f"{psd_results[name]['slope']:>12.3f} "
          f"{temporal_results[name]['cv']:>13.4f} "
          f"{prnu_results[name]['correlation']:>12.4f} "
          f"{dct_results[name]['kurtosis']:>10.2f}")
print("="*70)
print("\nExpected pattern if signals are clean:")
print("  PSD Slope:   Real closer to -2.5  |  AI more negative (steeper rolloff)")
print("  Temporal CV: Real higher           |  AI lower (smoother transitions)")
print("  PRNU Corr:   Real positive         |  AI near zero (no camera fingerprint)")
print("  DCT Kurt:    Difference depends on codec and content — observe the gap")
print("\n⚠️  Note: Heavy compression or screen-recording can weaken PRNU and PSD signals.")
print("   If signals are unclear, document that — it's still a valid finding.")

## Cell 12 — Copy All Outputs to Drive

In [ ]:
import shutil

output_drive_folder = "/content/drive/MyDrive/forensics_videos/results"
os.makedirs(output_drive_folder, exist_ok=True)

for fname in ["layer1_power_spectrum.png", "layer2_temporal.png",
              "layer3_prnu.png", "layer4_dct.png", "forensic_dashboard.png"]:
    src = f"/content/{fname}"
    if os.path.exists(src):
        shutil.copy(src, output_drive_folder)
        print(f"✅ Saved to Drive: {fname}")
    else:
        print(f"⚠️  Not found: {fname} (run the cell above first)")

print(f"\n📁 All outputs in: {output_drive_folder}")